# Türkçe arama modeli eğitimi (GPU)

Bu defter `intfloat/multilingual-e5-small` modelini Türkçe MS MARCO verisiyle ince ayarlar, ölçer,
tarayıcı formatına çevirir ve sonucu GitHub release'ine yükler.

**Önce:** Menüden *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seçin.
Sonra hücreleri sırayla çalıştırın (her hücrenin solundaki ▶ düğmesi veya Shift+Enter).

In [ ]:
!nvidia-smi -L || echo "GPU yok! Çalışma zamanı türünü T4 GPU yapın."
!git clone -q https://github.com/gunser-oz/gunser-oz-arama-modelleri.git
%cd gunser-oz-arama-modelleri
!pip install -q -r requirements.txt onnxruntime
!curl -sL -o mmarco_tr.jsonl.gz https://raw.githubusercontent.com/gunser-oz/gunser-oz-arama-modelleri/main/egitim/veri/mmarco_tr_100000.jsonl.gz && gunzip -f mmarco_tr.jsonl.gz
!wc -l mmarco_tr.jsonl

## 1. Eğitim

95 bin satır, 1 epoch, T4 GPU'da yaklaşık 10 dakika. Son 5 bin satır teste ayrılır.

In [ ]:
!python egitim/egit.py --veri mmarco_tr.jsonl --model intfloat/multilingual-e5-small \
  --cikti /content/cikti --test-ayir 5000 --epoch 1 --batch 64 --maks-uzunluk 192 \
  --dogrulama-orani 0.01 --kaydet-adim 500 --fp16
!cat /content/cikti/final/egitim_ozeti.json

## 2. Değerlendirme

Temel model ve yeni model aynı test kümesinde karşılaştırılır.

In [ ]:
!python egitim/degerlendir.py --veri mmarco_tr.jsonl --test 5000 --sorgu 1000 \
  --model intfloat/multilingual-e5-small --model /content/cikti/final --rapor /content/cikti/degerlendirme.json

## 3. Tarayıcı modeli (ONNX, 8 bit)

In [ ]:
!python egitim/onnx_aktar.py --model /content/cikti/final --cikti /content/cikti/web_model
!cp modeller/intfloat-multilingual-e5-small/tokenizer.json modeller/intfloat-multilingual-e5-small/tokenizer_config.json \
    modeller/intfloat-multilingual-e5-small/special_tokens_map.json /content/cikti/web_model/ 2>/dev/null || \
  python -c "from huggingface_hub import hf_hub_download as d; [d('intfloat/multilingual-e5-small', f, local_dir='/content/cikti/web_model') for f in ('tokenizer.json','tokenizer_config.json','special_tokens_map.json')]"
!rm -f /content/cikti/web_model/onnx/model.onnx
!ls -la /content/cikti/web_model/onnx

## 4. GitHub'a yükle

GitHub token gerekir: https://github.com/settings/tokens/new adresinde *repo* yetkisi işaretli bir token oluşturun.
Aşağıdaki hücre çalışınca token'ı sorar; yapıştırıp Enter'a basın (ekranda görünmez).

In [ ]:
import getpass, json, os, subprocess, urllib.request
token = getpass.getpass("GitHub token: ")
depo = "gunser-oz/gunser-oz-arama-modelleri"; tag = "model-gpu"
os.chdir("/content/cikti")
subprocess.run("rm -rf final/checkpoint-* && zip -qr model_gpu.zip final degerlendirme.json && zip -qr web_model_gpu.zip web_model", shell=True, check=True)
bas = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"}
def istek(url, veri=None, bas=bas, yontem=None):
    r = urllib.request.Request(url, data=veri, headers=bas, method=yontem)
    try:
        with urllib.request.urlopen(r) as y: return json.loads(y.read() or b"{}")
    except urllib.error.HTTPError as e:
        if e.code == 404: return None
        raise
rel = istek(f"https://api.github.com/repos/{depo}/releases/tags/{tag}")
if not rel:
    rel = istek(f"https://api.github.com/repos/{depo}/releases", json.dumps({"tag_name": tag, "name": "GPU'da eğitilmiş Türkçe model"}).encode())
for a in rel.get("assets", []):
    if a["name"] in ("model_gpu.zip", "web_model_gpu.zip"):
        istek(a["url"], yontem="DELETE")
for ad in ("model_gpu.zip", "web_model_gpu.zip"):
    veri = open(ad, "rb").read()
    b2 = dict(bas, **{"Content-Type": "application/zip", "Content-Length": str(len(veri))})
    istek(f"https://uploads.github.com/repos/{depo}/releases/{rel['id']}/assets?name={ad}", veri, b2)
    print("Yüklendi:", ad, round(len(veri)/1e6, 1), "MB")
print("Tamam. Claude'a 'model yüklendi' yazın.")